# MountainCarContinuous-v0 - REINFORCE (Colab + Gradio)

Runs in **Google Colab** after upload.
- Continuous action: engine thrust [-1, 1]
- **REINFORCE** (Policy Gradient): Gaussian policy μ(s)=tanh(w@state), explore with std σ
- Collect by episode → gradient ascent on return
- **GPU/CPU supported** (NumPy-based, runs on CPU; works on Colab CPU runtime)

## 1. Install libraries

In [ ]:
!pip install -q gymnasium[classic_control] gradio matplotlib

## 2. Reward shaping, Gaussian policy, and REINFORCE training

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pkg_resources is deprecated.*")

import gymnasium as gym
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gradio as gr


class PositionRewardShaping(gym.Wrapper):
    def __init__(self, env, scale=0.5):
        super().__init__(env)
        self.scale = scale

    def step(self, action):
        obs, reward, term, trunc, info = self.env.step(action)
        position = float(obs[0])
        reward = reward + self.scale * position
        return obs, reward, term, trunc, info


class GaussianPolicyREINFORCE:
    """Gaussian policy for continuous action: μ(s)=tanh(w@[1,s]), a ~ N(μ, σ), clip(a,-1,1)."""
    def __init__(self, obs_dim, action_dim, seed=None, init_scale=0.2, sigma=0.5):
        self.obs_dim = obs_dim
        self.action_dim = action_dim
        self.sigma = max(sigma, 1e-6)
        rng = np.random.default_rng(seed)
        self.w = rng.standard_normal(obs_dim + 1) * init_scale

    def _state_vec(self, state):
        return np.concatenate([[1.0], np.asarray(state, dtype=np.float64)])

    def mean(self, state):
        x = self._state_vec(state)
        return np.tanh(self.w @ x)

    def get_action(self, state, deterministic=False):
        mu = self.mean(state)
        if deterministic:
            return np.array([np.clip(mu, -1.0, 1.0)], dtype=np.float32)
        a = mu + self.sigma * np.random.standard_normal()
        return np.array([np.clip(a, -1.0, 1.0)], dtype=np.float32)

    def log_prob(self, state, action):
        mu = self.mean(state)
        a = float(action.flatten()[0])
        # N(mu, sigma^2) log prob (approx before clipping)
        return -0.5 * ((a - mu) / self.sigma) ** 2 - np.log(self.sigma) - 0.5 * np.log(2 * np.pi)

    def grad_log_prob(self, state, action):
        """∇_w log π(a|s)."""
        x = self._state_vec(state)
        z = self.w @ x
        mu = np.tanh(z)
        a = float(action.flatten()[0])
        d_log_pi_d_mu = (a - mu) / (self.sigma ** 2)
        d_mu_d_z = 1.0 - np.tanh(z) ** 2
        return d_log_pi_d_mu * d_mu_d_z * x

    def set_params(self, w):
        self.w = np.asarray(w, dtype=np.float64).ravel()

    def get_params(self):
        return self.w.copy()


def evaluate_policy(env, policy, n_episodes=100):
    rewards = []
    for _ in range(n_episodes):
        state, _ = env.reset()
        total = 0
        while True:
            action = policy.get_action(state, deterministic=True)
            state, reward, term, trunc, _ = env.step(action)
            total += reward
            if term or trunc:
                break
        rewards.append(total)
    return sum(rewards) / len(rewards)


def train_reinforce(env, policy, n_iterations=400, n_episodes_per_update=10,
                    lr=0.01, gamma=0.99, seed=None):
    eval_returns = []
    for it in range(n_iterations):
        trajectories = []
        for _ in range(n_episodes_per_update):
            state, _ = env.reset()
            traj = {"states": [], "actions": [], "rewards": [], "grads": []}
            while True:
                action = policy.get_action(state, deterministic=False)
                grad = policy.grad_log_prob(state, action)
                ns, reward, term, trunc, _ = env.step(action)
                traj["states"].append(state)
                traj["actions"].append(action)
                traj["rewards"].append(reward)
                traj["grads"].append(grad)
                state = ns
                if term or trunc:
                    break
            trajectories.append(traj)

        # Return G_t per step (discounted)
        all_returns = []
        for traj in trajectories:
            R = 0
            returns_t = []
            for r in reversed(traj["rewards"]):
                R = r + gamma * R
                returns_t.append(R)
            returns_t.reverse()
            all_returns.extend(returns_t)

        baseline = np.mean(all_returns)
        grad_w = np.zeros_like(policy.w)
        idx = 0
        for traj in trajectories:
            for t in range(len(traj["rewards"])):
                G = all_returns[idx]
                adv = G - baseline
                grad_w += adv * traj["grads"][t]
                idx += 1
        grad_w /= max(len(all_returns), 1)
        policy.w += lr * grad_w

        if (it + 1) % 10 == 0 or it == 0:
            mean_ret = evaluate_policy(env, policy, n_episodes=20)
            eval_returns.append(mean_ret)
        else:
            eval_returns.append(eval_returns[-1] if eval_returns else 0.0)

    return eval_returns


def run_mountaincar():
    env = PositionRewardShaping(gym.make("MountainCarContinuous-v0"), scale=0.5)
    obs_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    policy = GaussianPolicyREINFORCE(obs_dim, action_dim, sigma=0.5)

    mean_before = evaluate_policy(env, policy, n_episodes=100)
    eval_returns = train_reinforce(env, policy, n_iterations=400, n_episodes_per_update=10, lr=0.02, gamma=0.99)
    mean_after = evaluate_policy(env, policy, n_episodes=100)
    env.close()

    env_raw = gym.make("MountainCarContinuous-v0")
    mean_raw = evaluate_policy(env_raw, policy, n_episodes=100)
    env_raw.close()

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(eval_returns, alpha=0.4, color="blue", label="Eval Return (per 10 iter)")
    w = min(50, len(eval_returns) // 2)
    if w >= 2:
        ma = np.convolve(eval_returns, np.ones(w) / w, mode="valid")
        ax.plot(range(w - 1, len(eval_returns)), ma, color="red", label=f"Moving Avg ({w})")
    ax.set_title("REINFORCE - MountainCarContinuous-v0 (Position Reward Shaping)")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Mean Return")
    ax.legend()
    ax.grid(True)
    plt.tight_layout()

    txt = f"""Mean reward before (100 ep, shaped): {mean_before:.1f}
Mean reward after (100 ep, shaped): {mean_after:.1f}
Improvement: {mean_after - mean_before:+.1f}

Raw env mean reward (100 ep): {mean_raw:.1f} (+100 on goal)"""
    return fig, txt

## 3. Run Gradio app

### REINFORCE hyperparameter guide

| Parameter | Description |
|----------|------|
| **n_iterations** | Total update iterations |
| **n_episodes_per_update** | Episodes per update |
| **lr** | Policy learning rate |
| **gamma** | Discount (for return) |
| **sigma** | Gaussian policy exploration std |

In [ ]:
SLIDER_GUIDANCE = """
### REINFORCE hyperparameter guide

| Parameter | Description |
|----------|------|
| **n_iterations** | Total update iterations |
| **n_episodes_per_update** | Episodes per update |
| **lr** | Policy learning rate |
| **gamma** | Discount (for return) |
| **sigma** | Gaussian policy exploration std |
"""

with gr.Blocks(title="REINFORCE - MountainCar") as demo:
    gr.Markdown("# REINFORCE - MountainCarContinuous-v0")
    gr.Markdown("Gaussian policy + REINFORCE. Success = reach goal (+100).")
    gr.Markdown(SLIDER_GUIDANCE)

    run_btn = gr.Button("Train and show results", variant="primary")
    plot_out = gr.Plot(label="Learning curve")
    text_out = gr.Textbox(label="Results", lines=7)

    run_btn.click(fn=run_mountaincar, outputs=[plot_out, text_out])

demo.launch(share=False)  # share=True for public URL